## importing librarirs and load data

In [57]:
import numpy as np 
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt
import warnings
from sklearn.preprocessing import MinMaxScaler,OrdinalEncoder
from tensorflow.keras import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import L2
# from keras_tuner import RandomSearch
from sklearn.metrics import mean_squared_error as mse
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')


In [58]:
df=pd.read_excel(r"/kaggle/input/online-retail-uci/Online Retail.xlsx")

## Data Preprocessing & Feature engineering


In [59]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [60]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [61]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386048,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


In [62]:
df = df[~df['InvoiceNo'].astype('str').str.startswith('C')]

In [63]:
print(df.duplicated().sum() , "duplicated value")
df = df.drop_duplicates()

5231 duplicated value


## handiling missing values

In [64]:
max_id = df.CustomerID.max()
null_ids = df.CustomerID.isna()
null_ids = null_ids[null_ids == True]
df.loc[null_ids.index,'CustomerID'] = np.arange(max_id + 1,max_id + null_ids.sum() + 1)

In [65]:
df.CustomerID.isna().sum()

np.int64(0)

In [66]:
print(df.isna().sum().sum())
df.dropna(inplace=True)

1454


In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 525936 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    525936 non-null  object        
 1   StockCode    525936 non-null  object        
 2   Description  525936 non-null  object        
 3   Quantity     525936 non-null  int64         
 4   InvoiceDate  525936 non-null  datetime64[ns]
 5   UnitPrice    525936 non-null  float64       
 6   CustomerID   525936 non-null  float64       
 7   Country      525936 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 36.1+ MB


In [68]:
df=df[df['Quantity']>0]
df = df[df['UnitPrice'] > 0]

In [69]:
df['total_price']=df['Quantity']*df['UnitPrice']

In [70]:
df['total_price'] = df['total_price'].clip(lower=0)

### Create RFM features: 

In [71]:
# create RFM Recency Frequency Monetary features
snapshot_date=df['InvoiceDate'].max()+ pd.Timedelta(days=1)
df['Recency']=df.groupby('CustomerID')['InvoiceDate'].transform(lambda x:(snapshot_date-x.max()).days)

df['Frequency']=df.groupby('CustomerID')['InvoiceNo'].transform('nunique')

df['Monetary']=df.groupby('CustomerID')['total_price'].transform('sum')

## Create CLV

#### before CLV

In [72]:
df['AverageOrderValue'] = df['Monetary'] / df['Frequency']

In [73]:
df['FirstPerchase'] = df.groupby('CustomerID')['InvoiceDate'].transform(lambda x: (x.min()))
df['CustomerAge'] = (snapshot_date - df['FirstPerchase']).dt.days
df['CustomerAge'] = (df['CustomerAge'] - df['Recency'] )/30
df['CustomerAgeMonth'] = df['CustomerAge']/30

In [74]:
df['FrequencyPerMonth'] = df['CustomerAgeMonth'] / df['Frequency']

In [75]:
df['RecencyMonth'] = df['Recency']/30

In [76]:
df['CLV'] = df['FrequencyPerMonth'] * df['AverageOrderValue'] * 3 # 3 is customer lifespan
df['CLV'] = df['CLV'] * (1/( df['RecencyMonth']+1)) # make the CLV related to new customers

## Preprocessing

In [77]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,total_price,Recency,Frequency,Monetary,AverageOrderValue,FirstPerchase,CustomerAge,CustomerAgeMonth,FrequencyPerMonth,RecencyMonth,CLV
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,372,34,5391.21,158.565,2010-12-01 08:26:00,0.066667,0.002222,0.000065,12.4,0.00232
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,372,34,5391.21,158.565,2010-12-01 08:26:00,0.066667,0.002222,0.000065,12.4,0.00232
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,372,34,5391.21,158.565,2010-12-01 08:26:00,0.066667,0.002222,0.000065,12.4,0.00232
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,372,34,5391.21,158.565,2010-12-01 08:26:00,0.066667,0.002222,0.000065,12.4,0.00232
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,372,34,5391.21,158.565,2010-12-01 08:26:00,0.066667,0.002222,0.000065,12.4,0.00232


In [78]:
# we cant enter the InvoiceN,CustomerID and InvoiceDate to the model , aslo we Monetary is same as CLV so we will drop it
# also StockCode is not useful in the prediction
df.drop(columns=['InvoiceNo','InvoiceDate','StockCode','CustomerID','CustomerAge','FirstPerchase'],inplace=True)


In [79]:
df.head()

,Description,Quantity,UnitPrice,Country,total_price,Recency,Frequency,Monetary,AverageOrderValue,CustomerAgeMonth,FrequencyPerMonth,RecencyMonth,CLV
0,WHITE HANGING HEART T-LIGHT HOLDER,6,2.55,United Kingdom,15.30,372,34,5391.21,158.565,0.002222,0.000065,12.4,0.00232
1,WHITE METAL LANTERN,6,3.39,United Kingdom,20.34,372,34,5391.21,158.565,0.002222,0.000065,12.4,0.00232
2,CREAM CUPID HEARTS COAT HANGER,8,2.75,United Kingdom,22.00,372,34,5391.21,158.565,0.002222,0.000065,12.4,0.00232
3,KNITTED UNION FLAG HOT WATER BOTTLE,6,3.39,United Kingdom,20.34,372,34,5391.21,158.565,0.002222,0.000065,12.4,0.00232
4,RED WOOLLY HOTTIE WHITE HEART.,6,3.39,United Kingdom,20.34,372,34,5391.21,158.565,0.002222,0.000065,12.4,0.00232


In [80]:
# description does not have alot values, so can count it as categorical column
df.Description.nunique()

4026

In [81]:
df.duplicated().sum()

np.int64(80206)

In [82]:
# The duplicated is 70k from 520k its not a big number, if we didn't drop it 
# this will effect the model and make it overfit so we will drop it
df.drop_duplicates(inplace=True)

In [83]:
df = df.reset_index().drop('index',axis=1)

In [84]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 444672 entries, 0 to 444671
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Description        444672 non-null  object 
 1   Quantity           444672 non-null  int64  
 2   UnitPrice          444672 non-null  float64
 3   Country            444672 non-null  object 
 4   total_price        444672 non-null  float64
 5   Recency            444672 non-null  int64  
 6   Frequency          444672 non-null  int64  
 7   Monetary           444672 non-null  float64
 8   AverageOrderValue  444672 non-null  float64
 9   CustomerAgeMonth   444672 non-null  float64
 10  FrequencyPerMonth  444672 non-null  float64
 11  RecencyMonth       444672 non-null  float64
 12  CLV                444672 non-null  float64
dtypes: float64(8), int64(3), object(2)
memory usage: 44.1+ MB


In [85]:
# all the categorical columns are Ordinal
obj_cols = df.select_dtypes('object').columns
encoder = OrdinalEncoder()
df[obj_cols] = encoder.fit_transform(df[obj_cols])


In [86]:
df.head()

,Description,Quantity,UnitPrice,Country,total_price,Recency,Frequency,Monetary,AverageOrderValue,CustomerAgeMonth,FrequencyPerMonth,RecencyMonth,CLV
0,3844.0,6,2.55,36.0,15.30,372,34,5391.21,158.565,0.002222,0.000065,12.4,0.00232
1,3852.0,6,3.39,36.0,20.34,372,34,5391.21,158.565,0.002222,0.000065,12.4,0.00232
2,888.0,8,2.75,36.0,22.00,372,34,5391.21,158.565,0.002222,0.000065,12.4,0.00232
3,1859.0,6,3.39,36.0,20.34,372,34,5391.21,158.565,0.002222,0.000065,12.4,0.00232
4,2849.0,6,3.39,36.0,20.34,372,34,5391.21,158.565,0.002222,0.000065,12.4,0.00232
